# Getting opsim run and summary metric data

<img align="left" src = https://project.lsst.org/sites/default/files/Rubin-O-Logo_0.png width=250 style="padding: 10px"> 
<b>Getting opsim run and summary metric data</b> <br>
Contact authors: Eric Neilsen, Lynne Jones, Peter Yoachim<br>
Questions welcome at <a href="https://community.lsst.org/c/sci/survey-strategy">community.lsst.org/c/sci/survey-strategy</a> and the <a href="https://lsstc.slack.com/archives/C2LTWTP5J">#sims_operations</a> slack channel.<br>
Find additional MAF documentation and resources at <a href="https://rubin-sim.lsst.io">rubin-sim.lsst.io</a>. <br>

**Credit:** Stylistic elements of these notebooks were guided by the DP0.1 notebooks developed by Melissa Graham and the Rubin Observatory Community Engagement Team.

Note! This notebook was developed for evaluating the v2 simulations specifically. The content relating to comparing metrics with the archive plotting methods is still relevant, however we have not produced "family" json files for the newer v3 simulations. Summary data files for v3 are available however, in the relevant s3df download locations (such as https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs3.3/maf/summary.h5). 

In [ ]:
import os
from os import path

path_topdir = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"path_topdir = {path_topdir}")
file_mafresults_summary = os.path.join(path_topdir, "maf/fbs5.3/summary.h5")

In [ ]:
import rubin_sim
from rubin_sim import maf
# from rubin_sim.data import get_baseline

try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline

In [ ]:
opsim_fname = get_baseline()
run_name = path.splitext(path.basename(opsim_fname))[0]
print(f"Using {run_name}, to be read from {opsim_fname}")

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="04_getting_mafdata_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

## 1 Basic concepts

The Rubin Observatory Survey Strategy Team is producing an extensive collection of survey strategy simulations (using `opsim`) and corresponding analysis (using `MAF`). Many of these are of interest for science collaborations, and are publicly available. One interface to this data is [interactive web page](http://astro-lsst-01.astro.washington.edu:8080/) with lists of simulation runs and links ot `opsim` configuration and output files (databases of scheduled visits with simulated data quality) and output of `MAF` including summary values and plots:

- http://astro-lsst-01.astro.washington.edu:8080/
    
A programmatic interface to this data is also sometimes helpful. The `rubin_sim.maf.archive` module provides such an interface.

Survey strategy executions and analysis are assigned names and collected into groups for easy management and reference, according to the following nomenclature:

| term | discussion |
|------|------------|
| run | A **run** is a single execution of `opsim`. Each run produces an SQLite database of visits with data describing each visit (e.g. the start time, filter used, simulated seeing, etc.). Each `run` has a canonical "run name". Examples include `baseline_nexp2_v1.7.1_10yrs`, `baseline_v2.0_10yrs`, and `north_stripe_v2.0_10yrs`.|
| family | The survey strategy team often produces collections of runs designed to explore a specific aspect of survey strategy. Different runs in a collection vary the aspect of survey strategy being studied, while keeping other aspects the same. Comparing runs that are part of the same collection or **family** therefore supports exploring the effects of varying a specific parameter or other scheduler feature. Other collections for which direct comparison might be useful can also sometimes be grouped into the same family. Examples include the `baseline` family, which include simulations that have been used as "baselines" at different points in time; and `triplets`, which are runs that experimented with different methods of adding a third observation within a night. |
| summary metric | A **summary metric** is a single scalar representing some feature of an `opsim` run, generally one that indicates some aspect of the quality of the survey. Each `MAF` metric may produce any number of summary metrics, and each execution of MAF may construct an arbitrary number of summary metrics, depending on the MAF metric bundles executed. Each summary metric has a cannonical name derived from various elements of the metric bundle. |
| summary metric sets | Standard executions of MAF on opsim runs produce thousands of summary metrics, and users will only wish to inspect and compare limited subsets of these summary metrics at any given time. The survey strategy team has therefore pre-defined a collection of named sets of metrics, so that sets of metrics usefully examined as a group can be referenced together. Examples of summary metric sets include `SRD` (which correspond to requirement in the Science Requirements Document), `WFd Depths` (which describes the number of visits and typical depths in the WFD region), `TVS KNe` (summary metrics related to KNe relevant for the transients and variable stars working group), `DESC WFD` (summary metrics of interest to DESC analysis of the WFD survey), and more. | 

- author of corrections : Sylvie Dagoret-Campagne
- creation date : 2026-08-04
- copied and adapted (corrected) from https://github.com/lsst/rubin_sim_notebooks/tree/main/maf/tutorial

## 2 Notebook preparation

The following is a development style aid; only uncomment if developing the notebook:

In [ ]:
# %load_ext nb_black
# %load_ext pycodestyle_magic
# %flake8_on --ignore E501,W505

Required imports:

In [ ]:
from rubin_sim import maf
from rubin_sim.maf.run_comparison import archive

## 3 Run families

The "families" json file organizes `opsim` runs into "families," groups of runs that vary in a controlled way, and which are approprate for direct comparison with each other in order to understand the effects of varying specific parameters, or making specific alterations to survey strategy. Note that there are different versions of this file for v1 and v2 of the opsim outputs.

You can download a table of families, their descriptions, and definitions into a `pandas.DataFrame` thus:

In [ ]:
families = maf.get_family_descriptions()
families

By default, `get_family_descriptions` retrives the runs data from a json file at the URL provided in `archive.FAMILY_SOURCE`:

In [ ]:
archive.FAMILY_SOURCE

If you wish to load runs from an alternate source, it can be specified with the `family_source` argument to `get_family_descriptions`.

You can use the loaded data to get a list of available families:

You can get more pleasantly formatted descriptions of the families using `archive.describe_families`:

In [ ]:
maf.describe_families(families.loc[["baseline", "technical"]])

## 4 Getting a table of runs

You can download a `pandas.DataFrame` of runs with basic information on each run using `get_runs`:

In [ ]:
runs = maf.get_runs()
runs

By default, `get_runs` retrives the runs data from the same json file as `archive.get_family_descriptions`, and also has an argument to download the data from a different source.

`get_family_descriptions` and `get_runs` load the same data, but the former is indexed by families, with one row per family; and the later by runs, with one row per run. In the former case, values that vary by run for the same family have list values, while in the later case values that vary by family for the same run have list values.

If you want a `DataFrame` with one row per run/family combination, such that there are no columns with list-valued cells, you can `explode` the `pandas.DataFrame` returned by `get_runs` (or `get_family_descriptions`) and set the `family` column to be the index, or use the `get_family_runs` shorthand:

In [ ]:
family_runs = maf.get_family_runs()
family_runs

This makes it easy to reference just the runs from a family (or set of families) of interest.

`get_family_runs` reads the run metadata from the same source as `get_runs`, and (like `get_runs`) alternate sources can be specified by an argument.

First, let's look at a list of all families, and how many runs are in each:

In [ ]:
family_runs.groupby("family").agg({"run": "count"})

If I want to work with just runs in `baseline` or `rolling`, I can build `pandas.DataFrame` of such runs by slicing `family_runs`:

In [ ]:
my_runs = family_runs.loc[["baseline", "rolling"]]
my_runs

## 5 Getting summary metrics on runs

`get_metric_summaries` will retrieve the "summary" results of MAF for these runs into a `pandas.DataFrame`. By default, summary data is downloaded from the URL specified by `archive.DEFAULT_SUMMARY_SOURCE`. Users may pass an alternate source (URL or file name) to `get_metric_summaries` to load the data from elsewhere.
Note that there may be multiple versions of this summary file available; using the latest version is generally the best choice. 

In [ ]:
archive.SUMMARY_SOURCE

In general, it's worthwhile to download the entire summary file once and subset it later:

In [ ]:
# summaries = maf.get_metric_summaries()

In [ ]:
from rubin_sim.maf.run_comparison.archive import get_metric_summaries

In [ ]:
df = get_metric_summaries(summary_source=file_mafresults_summary)

In [ ]:
df

Un fichier summary.h5 contient toute l'archive MAF.

Pour un seul run, il peut y avoir :

- profondeur : CoaddM5
- nombre de visites : Count
- seeing : Median seeing
- temps : Year1Count
- cosmologie : SNNSNMetric, TDC, weak lensing
-

- KBO

- NEO

- ToO

etc.

Donc plusieurs milliers de colonnes sont normales.

But, it is also possible to get only metrics on runs in a single (or a few) families by supplying arguments to specify which runs, run families, and metrics you want.  (note that this downloads the entire dataframe and then only returns the subset -- if doing this frequently, just download it once). 

In [ ]:
# maf.get_metric_summaries(run_families=["rolling", "baseline"])

It returned more than 10000 metrics for these runs, which is more than necessary or convenient for most purposes (but are provided in order to allow for individual exploration).  The next section highlights some aids to that exploration. 

## 6 Metric sets

Rather than sort through all these metrics, you can work with pre-defined sets of metrics generated for a variety of purposes. `get_metric_sets` loads definitions of sets of metrics (and other metric metadata) from a URL specified by `archive.DEFAULT_METRIC_SET_SOURCE`. Users can load this data by passing the URL or file name as the argument to `get_metric_sets`. *Metric set definitions cannot be arbitrarily mixed with summary sources: each version of summary source must be matched with a corresponding metric set source.*

In [ ]:
archive.METRIC_SET_SOURCE

You can get the set definitions using `get_metric_sets`:
(again, this can be helpful to download once)

In [ ]:
metric_sets = maf.get_metric_sets()
metric_sets

You can see what 'groups' of metrics have already been defined.

In [ ]:
list(metric_sets.groupby("metric set").first().index)

Slicing this `pandas.DataFrame` will give you the metrics for just the sets you specify:

In [ ]:
metric_sets.loc["SRD"]

In [ ]:
metric_sets.loc[["SRD", "SCOC"], :]

Writing new metric sets is extremely reasonable -- and there are some functions in the `archive` module to help make this easier (see `create_metric_set_df` and `write_metric_sets`). 

You can get metric summaries by getting the list of metrics from the `DataFrame` through the `metrics` option, or you can just set the `metric_sets` option directly:

In [ ]:
maf.get_metric_summaries(summary_source=summaries, run_families="rolling", metric_sets="SCOC")

## 7 Normalizing summary metrics

When comparing many runs with many metrics, it helps if each metric behaves similarly. As recorded, though, the numeric values of different metrics mean different things. For example, some metrics are better when they have higher values, others are better with lower values. Furthermore, they are all scaled differently.

For ease of comparison, we can transform all of them such that they take a value of 1 if they are equally good to some baseline, positive if better, negative if worse.

This comparison continues to be rough: different metrics continue to have different units, and so are not directly comparable. Still, it is a rough improvement.

Pick a family of runs and a set of metrics to use in this example:

In [ ]:
this_family = ["baseline", "rolling"]
this_metric_set = "SRD"

We need to pick a reference run to define to have a normalized value of 1:

In [ ]:
baseline_run = "baseline_v2.0_10yrs"

Now get all the (unnormalized) metrics summary values:

In [ ]:
summary = maf.get_metric_summaries(
    summary_source=summaries, run_families=this_family, metric_sets=this_metric_set
)
summary

Normalize it and look at the results. We pass in the metric_set itself as this contains information on *HOW* to normalize the metric values (e.g. do they need to be inverted because "bigger" is worse? or do they represent magnitude values, so they should be subtracted instead of divided?)

In [ ]:
mset = metric_sets.loc[this_metric_set]
norm_summary = maf.normalize_metric_summaries(baseline_run, summary, metric_sets=mset)
norm_summary

(note how the normalized values for proper motion and parallax uncertainty are smaller when the uncertainties themselves are larger, while the median number of visits per point (fONv MedianNvis) is larger when the value is larger -- as appropriate). 

## 8 Cartesian plots

This would be easier to interpret if presented graphically, for which you can use `plot_run_metric`:

In [ ]:
maf.plot_run_metric(summary, baseline_run=baseline_run, metric_set=metric_sets.loc[this_metric_set])

Passing the metric_set to plot_run_metric adds some default styling for the colors, etc., but also (again) adds the correct normalization. 

The labels on this plot are long and precise, but can be hard to read or show in a small area. Shorter names for both runs and metrics are available in the `pandas.DataFrames` we have already downloaded, and we can build transformations from these `DataFrame`s.

In [ ]:
metric_label_map = metric_sets.loc[this_metric_set, "short_name"]
metric_label_map

In [ ]:
run_label_map = family_runs.loc[this_family, ["run", "brief"]].set_index("run")["brief"]
run_label_map

These can be passed to `plot_metric_summary` to replace the labels:
(note that using substitute names can obfuscate the run name, in particular -- was this v1.7 or v2.0, etc -- in ways that make tracking down the contents of plots later difficult)

In [ ]:
maf.plot_run_metric(
    summary,
    baseline_run=baseline_run,
    run_label_map=run_label_map,
    metric_label_map=metric_label_map,
    metric_set=metric_sets.loc[this_metric_set],
    out_dir=data_dir,
)

The `vertical_quantity` and `horizontal_quantity` options will let you set which axis (horizontal, vertical, or color) is mapped to which quantity (run, metric name, metric value), and additional arguments set the color maps, markers, and line styles connecting the points:

In [ ]:
import matplotlib as mpl

this_metric_set = "SCOC"

this_family = ["baseline", "microsurveys"]
run_label_map = family_runs.loc[this_family, ["run", "brief"]].set_index("run")["brief"]
metric_label_map = metric_sets.loc[this_metric_set, "short_name"]

summary = archive.get_metric_summaries(this_family, this_metric_set)
metric_label_map = metric_sets.loc[this_metric_set, "short_name"]
maf.plot_run_metric(
    summary,
    baseline_run=baseline_run,
    metric_set=metric_sets.loc[this_metric_set],
    vertical_quantity="value",
    horizontal_quantity="run",
    run_label_map=run_label_map,
    metric_label_map=metric_label_map,
    cmap=mpl.cm.tab10,
    linestyles=["-", ":", "--", "-."],
    markers=["o", "v", "^", "<", ">", "*", "H", "D"],
)

In [ ]:
summary

## 9 Mesh plots

Alternately, you can color code the metric value itself using `plot_run_metric_mesh`:

In [ ]:
this_metric_set = ["WFD Depths", "cadence"]
metric_label_map = metric_sets.loc[this_metric_set, "short_name"].droplevel("metric set")

this_family = ["baseline", "rolling"]
run_label_map = family_runs.loc[this_family, ["run", "brief"]].set_index("run")["brief"]

summary = maf.get_metric_summaries(this_family, this_metric_set)

maf.plot_run_metric_mesh(
    summary,
    baseline_run=baseline_run,
    metric_set=metric_sets.loc[this_metric_set],
    run_label_map=run_label_map,
    metric_label_map=metric_label_map,
)

## 10 Radar plots

Finally, if the numbers of runs and metrics are manageable, you can compare different metrics of different runs with a radar plot.

Let's select a modest collection of metrics and family of runs, and build a summary:

In [ ]:
family_runs.loc[["baseline", "technical"], :]

In [ ]:
this_metric_set = "radar"
this_family = ["baseline", "technical"]
summary = maf.get_metric_summaries(this_family, this_metric_set)

The radar plot function requires that the data already be normalized, so normalize it:

In [ ]:
norm_summary = maf.normalize_metric_summaries(
    baseline_run, summary, metric_sets=metric_sets.loc[this_metric_set]
)

The radar plot function takes the run and metric names from the `DataFrame` row and column names, so we can use short name by renaming the rows and columns:

In [ ]:
metric_label_map = metric_sets.loc[this_metric_set, "short_name"]
run_label_map = family_runs.loc[this_family, ["run", "brief"]].set_index("run")["brief"]
norm_summary.rename(columns=run_label_map, index=metric_label_map, inplace=True)

Make the radar plot:

In [ ]:
fig, ax = maf.radar(norm_summary.T, bbox_to_anchor=(3, 0), rgrids=[0.7, 1.0, 1.3])

## 11 Plotting yet more metrics and runs

Multiple sets of metrics and families of runs can be retrieved and plotted at once, and these can be supplemented by additional individual runs and metrics:

In [ ]:
these_metric_sets = ["SRD", "WFD Depths", "radar"]
these_families = ["baseline", "triplets", "long gaps no pairs"]
extra_runs = [
    "noroll_v2.0_10yrs",
]
extra_metrics = [
    "Median Median Intra-Night Gap WFD HealpixSubsetSlicer",
]
summary = maf.get_metric_summaries(these_families, these_metric_sets, runs=extra_runs, metrics=extra_metrics)

Because the slicing in pandas will return a multilevel index when multiple runs or families are sliced on, a little additional processing is needed to get a mapping from run or index name alone:

In [ ]:
mset = metric_sets.loc[these_metric_sets].reset_index("metric set", drop=True, allow_duplicates=False)

In [ ]:
these_runs = list(summary.index)
run_label_map = family_runs[["run", "brief"]].set_index("run").loc[these_runs, "brief"].groupby("run").first()

In [ ]:
these_metrics = list(summary.columns)
metric_label_map = metric_sets.loc[(slice(None), these_metrics), "short_name"].groupby("metric").first()

In [ ]:
fig, ax = maf.plot_run_metric_mesh(
    summary,
    baseline_run=baseline_run,
    metric_set=mset,
    run_label_map=run_label_map,
    metric_label_map=metric_label_map,
)
fig.set_figwidth(13)
fig.set_figheight(8)

## 12 Plotting other metrics

Not all metrics present in the summary table have corresponding columns in the `metrics_set` data. 
If they do not, the MAF code does not know how to normalize these values (do they correspond to magnitudes? should they be inverted so that "bigger values are better" in the normalized plots?) and it's harder to deal with them.
For example, minimum WFD depth values have no normalization values:

In [ ]:
min_depth_metrics = tuple(f"Min CoaddM5 WFD {b} band HealpixSubsetSlicer" for b in "ugrizy")
min_depth_metrics

We can still load the un-normalied metric values:

In [ ]:
summary = maf.get_metric_summaries(
    summary_source=summaries,
    run_families=["baseline", "bluer balance"],
    metrics=min_depth_metrics,
    metric_order="set",
)
summary

In [ ]:
fig, ax = maf.plot_run_metric(summary, shade_fraction=None)
ax.set_xlim(21, 28)

If you want to properly normalize them, you can create you own metric_sets `DataFrame`:

In [ ]:
mset = maf.create_metric_set_df(
    metric_set="min depth wfd",
    metrics=min_depth_metrics,
    short_name=[f"Min {b} band depth WFD" for b in "ugrizy"],
    style=["c-", "g-", "y-", "r-", "m-", "k-"],
    mag=True,
)
mset

In [ ]:
maf.plot_run_metric(
    summary,
    baseline_run=baseline_run,
    metric_label_map=mset.loc["min depth wfd"]["short_name"],
    metric_set=mset.loc["min depth wfd"],
)

## 13 Running additional MAF metrics

If the summary metrics are inadequate for what you need, you can download the opsim databases using URLs found in the `runs` `DataFrame` we downloaded above (using `get_runs`):

In [ ]:
runs

## 14 Putting it all together to explore a set of simulations

When exploring a new set of simulations (e.g. version 2.0), it can be useful to load the relevant family, run, and metric set `DataFrame`s, and define a single shorthand function to give everything you want on specific families.

For example, for the v2.0 runs, we let's get the relevant `DataFrames` and define such a function:

In [ ]:
url_base = "https://raw.githubusercontent.com/lsst-pst/survey_strategy/main/fbs_2.0/"
families = maf.get_family_descriptions(url_base + "runs_v2.2.json")
summary = maf.get_metric_summaries(summary_source=url_base + "summary_2022_11_16.csv")
metric_sets = maf.get_metric_sets(url_base + "metric_sets.json")

Grab a common baseline, using the baseline family:

In [ ]:
baseline_run = families.loc["baseline", "run"][0]
baseline_run

Make a mesh that shows all runs, and all metrics that are members of at least one metric set:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Include metrics from all metric sets (use the metric set so that we can invert appropriately)
# skip DD specific metrics
non_dd = [ms for ms in list(metric_sets.groupby("metric set").first().index) if not "DD" in ms]
mset = metric_sets.loc[non_dd].reset_index(drop=True).drop("style", axis=1).drop("short_name", axis=1)
mset = mset.drop_duplicates().set_index("metric", drop=False, verify_integrity=True)

# Adjust the plot size so everything will fit without the labels overlapping
# Requires fiddling by hand.
fig, ax = plt.subplots(figsize=(30, 20))
maf.plot_run_metric_mesh(
    summary.loc[:, mset["metric"]],
    baseline_run=baseline_run,
    metric_set=mset,
    ax=ax,
)
fig.set_figheight(40)
fig.set_figwidth(15)